# What Comes Next

You just spent a session building an agent loop around a RAG pipeline. The loop evaluates its own retrieval, rewrites queries, selects tools, and declines to answer when it should. The eval scores are real scores. The traces are real traces.

And it was a few hours.

What follows is an honest map of what a production engagement with agentic RAG actually looks like.

## 1.1 What You Built Is a Mental Model

The lab was not designed to produce a deliverable. It was designed to give you felt experience with a problem space that your customers are trying to navigate, so you can ask better questions, spot bad assumptions earlier, and know when to escalate to the people who do the build.

Production is a different problem than the lab. Not a harder version of the same problem. A different one.

The difference is not the model or the agent loop. It is everything around them.

## 1.2 The Chain Still Applies

The Escalation Lab established that a RAG pipeline is a chain: bad ingestion produces bad chunks, bad chunks produce bad retrieval, bad retrieval means the model never sees the right information. The agent loop does not break this chain. It adds links to it.

A tool description that is vague produces wrong tool selection. A dispatcher that silently swallows errors produces traces that lie. An iteration budget that is too low produces premature best-effort answers. An iteration budget that is too high produces latency that users will not tolerate.

The agent loop gave the system the ability to recover from bad retrieval. It did not give the system the ability to recover from bad tool definitions, bad corpus quality, or bad architectural decisions. Those still require human judgment, and the chain is only as strong as its weakest link.

## 1.3 What the Lab Did Not Show You

The lab was designed to teach a workflow. It was not designed to simulate production. The following topics were intentionally omitted to keep the lab focused, but they are not optional in a real deployment.

### 1.3.1 Tool reliability at scale

The lab defined three tools. A production agent may have ten or twenty. As the tool set grows, the model's ability to select the right tool degrades unless the descriptions are maintained with the same discipline as API documentation. Tool descriptions are not comments. They are decision inputs. When two tools have overlapping descriptions, the model will route unpredictably between them. When a tool description drifts out of sync with the tool's actual behavior, the agent will select it for the wrong reasons and the trace will look correct while the answer is wrong.

The fix is governance: review tool descriptions the way you review API contracts. Test tool selection as part of CI. Track which tools are selected for which question categories and alert when the distribution shifts.

### 1.3.2 Multi-agent orchestration

The lab used a single agent loop with a single model. Production systems increasingly use multiple agents, each specialized for a different task: one for retrieval, one for calculation, one for summarization, one for routing. The agents communicate through structured messages and a shared state.

Multi-agent systems introduce coordination problems that single-agent systems do not have. Agents can disagree. Agents can loop. Agents can pass malformed state to each other. The trace becomes a graph instead of a list, and debugging requires tools that can visualize agent-to-agent communication, not just tool calls within a single loop.

This is the next level of architectural complexity beyond what the lab covered. Reach for it when a single agent loop cannot handle the diversity of tasks in your domain, not before.

### 1.3.3 Human-in-the-loop for tool decisions

The lab's agent loop was fully autonomous: it selected tools, executed them, and produced answers without human intervention. In high-stakes domains, that autonomy is a liability.

A production system may require human approval before executing certain tools, especially tools that modify state (writing to a database, sending a message, triggering a workflow). The agent proposes the tool call. A human reviews and approves or rejects it. The loop resumes.

This changes the loop from synchronous to asynchronous and introduces UX design problems: how do you present a tool call proposal to a non-technical user? How long do you wait for approval before timing out? What happens when the human rejects the proposal? These are design questions, not model questions, and they shape the architecture more than the model choice does.

### 1.3.4 Trace-based monitoring and alerting

The lab recorded traces for inspection. Production requires traces for monitoring. Every tool call, every retrieval result, every evaluation judgment, and every final answer should be logged in a structured format that supports querying and alerting.

The questions you need to answer from traces in production are different from the lab. How many queries per hour trigger the no_answer tool? Is that number increasing? What is the average number of retrieval retries before the agent finds sufficient context? Which tool descriptions are causing the most selection errors? What is the p95 latency for queries that require tool calls versus queries that the model answers directly?

Without this infrastructure, the agent loop is a black box in production. The traces exist but no one is watching them, and degradation accumulates until a user reports that the system used to work better.

### 1.3.5 Output guardrails

The lab focused on answer correctness. Production requires answer safety. A model connected to a customer corpus can leak personally identifiable information, generate hallucinated citations that look real, reference documents the user is not authorized to see, or produce answers that are technically grounded but contextually inappropriate.

The fix is a validation layer between the model and the user. Check outputs for PII patterns before returning them. Verify that cited sources actually exist in the corpus and that the user has access to them. Flag answers where the model's confidence is low or where the retrieved context does not support the claim. Consider a second, smaller model whose only job is to evaluate whether the primary model's output is safe to return.

### 1.3.6 Document freshness and re-ingestion

The lab treated the corpus as a fixed artifact. Production corpora are not fixed. Documents are updated, superseded, retracted, and versioned.

When a document changes, the old chunks are still in the retrieval index. Unless you have a re-ingestion pipeline that detects changes, re-processes affected documents, removes stale chunks, and validates that retrieval still returns current information, your system will silently serve outdated answers with full confidence.

The hard part is not the re-ingestion itself. It is knowing that a document changed. In enterprise environments, documents live in SharePoint, Confluence, S3 buckets, shared drives, and email attachments. Building a reliable change detection layer across those sources is often the most underestimated piece of the production architecture.

### 1.3.7 Access control in retrieval

In the lab, every participant could see every document. In production, that is almost never true. A RAG system that retrieves chunks without respecting document-level permissions is a data exfiltration tool with a friendly interface.

Access control must be enforced either at query time, by filtering retrieval results against the user's permissions before they reach the model, or at ingestion time, by maintaining separate indices per access tier. Both approaches have trade-offs. The conversation with the customer about this topic should happen early, not after the first security review.

### 1.3.8 Cost and latency budgets

The agent loop multiplies API calls. A single question may trigger 2-3 tool calls, each requiring a round trip to the model endpoint. In the lab, latency was irrelevant. In production, users expect sub-second responses, and each API call has a dollar cost.

Semantic caching addresses the most common case: many users ask variations of the same question. Rather than running the full agent loop every time, cache the response for semantically similar queries and return it directly. The implementation requires a similarity threshold for cache hits, a TTL strategy that accounts for corpus freshness, and invalidation logic that clears cached responses when the underlying documents change.

The iteration budget in the agent loop is also a cost lever. A max_retries of 2 means at most 3 retrieval calls per question. A max_retries of 5 is more thorough but 2-3x more expensive and slower. The right number depends on the cost of a wrong answer versus the cost of latency and compute.

### 1.3.9 User feedback loops

The evaluation set you built in the lab was authored by the people who built the system. In production, the most valuable signal about answer quality comes from the people who use the system every day.

A thumbs-up/thumbs-down mechanism on each response is the minimum. Better is a correction flow where users can flag a wrong answer and provide the right one. Best is a pipeline that routes those corrections back into the evaluation set, so the questions real users actually care about become the questions the system is measured against.

Without this loop, the evaluation set fossilizes. The system can score 10/10 on the original eval while silently failing on the questions users actually ask.

## 1.4 Common Anti-Patterns

The agentic RAG pattern is new enough that the failure modes are not yet well-cataloged. The following are mistakes that recur across engagements. Each one is easy to make and hard to detect without deliberate evaluation.

**Too many tools with overlapping descriptions.** A system with fifteen tools where three of them could plausibly handle a given question will produce inconsistent routing. The model is pattern-matching against descriptions, and overlapping descriptions create ambiguity. Start with the minimum tool set that covers your failure modes. Add tools only when evaluation shows a gap that no existing tool addresses.

**Agents that never abstain.** If the no_answer tool is never selected across your evaluation set, something is wrong. Either the tool description is too restrictive, or the eval set does not include out-of-scope questions. A system that always answers is a system that sometimes hallucinates. Test with questions the corpus cannot answer and verify that the agent declines.

**Retry loops that burn budget without improving results.** A max_retries of 5 is not better than 2 if the third, fourth, and fifth attempts are rewriting the query in circles. Monitor retry effectiveness: if the second attempt rarely improves on the first, a higher budget is wasted compute. Log the verdict at each attempt and track the rate at which retries actually change the outcome.

**Tool descriptions treated as comments.** A description that says "searches for stuff" will produce unreliable selection. A description that says "search the Basic Fantasy RPG rulebook corpus for text passages relevant to the query" will produce reliable selection. The lab demonstrated this in Section 3. In production, the failure mode is subtler: a description that was precise when it was written drifts out of sync as the tool's behavior changes. Tool descriptions need the same review discipline as API documentation.

**Traces that get logged but never monitored.** Recording every tool call, retrieval result, and evaluation judgment is necessary but not sufficient. If no one is watching the traces, degradation accumulates silently. A tool that starts getting selected for the wrong question category, a retrieval evaluator that starts returning "sufficient" for weak chunks, a retry count that creeps upward over time — none of these will announce themselves. Traces without monitoring are audit logs for an audit that never happens.

**Optimizing for accuracy instead of reliability.** A system that scores 9/10 on answer correctness looks better than one that scores 8/10. But if two of those nine correct answers came through wrong reasoning (the Lucky quadrant from Section 5), the 9/10 system is less reliable than the 8/10 system where all eight are in the Reliable quadrant. Accuracy is a necessary metric. It is not a sufficient one.

## 1.5 Testing Agent Systems in CI

The lab evaluated the agent loop manually: run the questions, inspect the traces, score the results. That works for a workshop. It does not work for a system that changes every sprint.

In production, the evaluation set and the 2x2 matrix should run in CI. A tool description change is a code change. A prompt change is a code change. A new tool added to the dispatcher is a code change. Each of these can shift tool selection across every question in the eval set, and the shift may not be obvious from reading the diff.

The minimum CI pipeline for an agentic system runs the full eval set on every PR that touches tool definitions, prompts, or the agent loop. It checks three things: answer correctness has not regressed, the Reliable quadrant count has not dropped, and no question has moved from Reliable to Lucky (which means the answer is still correct but the reasoning path broke). A PR that improves accuracy by one question but moves two questions from Reliable to Lucky is a net regression in reliability, even though the accuracy number went up.

Pin your eval set in version control alongside the tool definitions. When you add a new eval question, it should be a deliberate commit with a reason. When you remove one, same. The eval set is the contract between the team and the system: these are the questions we promise to get right, through the right reasoning path, every time.

This is not overhead. It is the difference between a system that works today and a system that still works after the next twenty PRs.

## 1.6 The Conversation to Have with Your Customer

When a customer says they want to build an agentic RAG system, the instinct is to talk about agents. The right conversation starts somewhere else.

Ask them: how are you measuring correct? And how are you measuring correct reasoning?

If they can only answer the first question, they will build a system that scores well on accuracy and fails unpredictably in production. The 2x2 matrix from Section 5 is the tool that makes this concrete: a correct answer from wrong reasoning is not a reliable system.

The evaluation set you build with their subject matter experts, even ten or twenty questions with agreed-upon reference answers and expected reasoning paths, is often the most productive artifact of the first month. It forces the customer to articulate standards they have never written down. It gives you a shared definition of done. And it gives every subsequent recommendation a defensible basis.

When they ask whether they need an agent loop, the answer that earns trust is not yes or no. It is: "Here is what the evaluation shows. Here are the questions failing and why. Here is what passive RAG cannot fix. Here is the specific failure mode an agent loop addresses." That positions you as someone who solves problems methodically, not someone who sells the most complex option.

## 1.7 Further Reading

### Agentic RAG: Core Papers

[Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection](https://arxiv.org/abs/2310.11511) (Asai et al., 2023) -- the paper most directly aligned with what this lab built. The model learns to decide when to retrieve, evaluates whether retrieval was useful, and critiques its own output. The evaluate-before-answering pattern from Section 2 is a simplified version of this approach.

[Corrective Retrieval Augmented Generation (CRAG)](https://arxiv.org/abs/2401.15884) (Yan et al., 2024) -- introduces a lightweight retrieval evaluator that triggers corrective actions when retrieval quality is low. The three-verdict evaluator in Section 2 (sufficient / partial / irrelevant) maps directly to CRAG's confidence scoring.

[Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity](https://arxiv.org/abs/2403.14403) (Jeong et al., 2024) -- instead of running the same pipeline for every question, the system classifies question complexity first and routes to different strategies. Relevant to the out-of-scope classifier discussed in Section 2's abstain note.

[ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (Yao et al., 2023) -- the reasoning-and-acting framework that underpins most agent loop implementations. The interleaved think-act-observe cycle is the conceptual foundation for the retrieve-evaluate-decide loop built in this lab.

### Tool Use and Function Calling

[Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761) (Schick et al., 2023) -- the foundational paper on LLMs learning to use external tools. Establishes that tool use is a capability, not a trick.

[Gorilla: Large Language Model Connected with Massive APIs](https://arxiv.org/abs/2305.15334) (Patil et al., 2023) -- demonstrates that tool selection accuracy depends heavily on tool description quality. The vague-vs-precise exercise in Section 3 is a hands-on version of this paper's central finding.

[Berkeley Function-Calling Leaderboard](https://gorilla.cs.berkeley.edu/leaderboard.html) -- a live benchmark tracking how well different models handle function calling. Useful for selecting a model when tool selection reliability is the primary concern.

### Query Rewriting and Retrieval Optimization

[Query Rewriting for Retrieval-Augmented Large Language Models](https://arxiv.org/abs/2305.14283) (Ma et al., 2023) -- formalizes the query rewriting step the agent loop performs in Section 2. Shows that rewriting the query based on retrieval feedback consistently improves answer quality.

[Interleaving Retrieval with Chain-of-Thought Reasoning](https://arxiv.org/abs/2212.10509) (Trivedi et al., 2023) -- demonstrates that multi-hop questions benefit from interleaving retrieval with reasoning steps rather than retrieving all context upfront. Directly relevant to the implicit reasoning failure mode from Section 1.

[Active Retrieval Augmented Generation](https://arxiv.org/abs/2305.06983) (Jiang et al., 2023) -- the model decides when and what to retrieve during generation, rather than retrieving everything before generating. The agent loop's conditional retrieval is a practical implementation of this idea.

### Evaluation and Reliability

[RAGAS: Automated Evaluation of Retrieval Augmented Generation](https://arxiv.org/abs/2309.15217) -- automated RAG evaluation metrics that do not require human-annotated ground truth. The answer correctness dimension from Section 5 aligns with RAGAS's answer relevancy metric.

[Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685) (Zheng et al., 2023) -- the paper behind using LLMs to evaluate other LLMs. Directly relevant to the model-as-judge approach used in Sections 4 and 5.

[Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172) -- explains why what you retrieve and where it appears in the prompt both matter. Performance degrades when relevant information appears in the middle of a long context.

### Agent Frameworks and Orchestration

[LangGraph documentation](https://langchain-ai.github.io/langgraph/) -- the graph-based agent orchestration framework from LangChain. Relevant when a single loop is not enough and you need multi-step, multi-agent workflows with conditional branching.

[Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) (Anthropic, 2024) -- practical guidance on agent design patterns, including when to use tool-calling agents versus prompt-chaining workflows. The key insight aligns with this lab: start with the simplest pattern that works, and only add complexity when evaluation justifies it.

[OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling) -- the API specification this lab's tool definitions follow. The guide includes best practices for writing tool descriptions that improve selection accuracy.

### Security and Governance

[OWASP Top 10 for Large Language Model Applications](https://owasp.org/www-project-top-10-for-large-language-model-applications/) -- the definitive catalog of LLM-specific vulnerabilities. In agentic systems, tool execution adds attack surface beyond prompt injection: a compromised tool description can cause the agent to execute unintended actions.

[Not what you've signed up for: Compromising Real-World LLM-Integrated Applications with Indirect Prompt Injection](https://arxiv.org/abs/2302.12173) (Greshake et al., 2023) -- demonstrates how prompt injection works in RAG systems, where retrieved documents become the attack vector. In agentic RAG, this risk compounds: injected content can influence tool selection, not just answer generation.

[NIST AI 600-1: AI Risk Management Framework for Generative AI](https://csrc.nist.gov/pubs/ai/600/1/final) -- the US federal framework for GenAI risk. The trace-based auditability built in Section 2 is a step toward the provenance requirements this framework describes.

### Books

[AI Engineering](https://www.oreilly.com/library/view/ai-engineering/9781098166298/) (Chip Huyen, O'Reilly) -- covers building AI applications with foundation models, including evaluation, adaptation, and deployment. The chapter on agents is directly relevant.

[Enterprise RAG](https://livebook.manning.com/book/enterprise-rag/welcome) (Tyler Suard and Darshil Modi, Manning) -- practical guidance on building RAG systems for enterprise environments, from corpus management to reducing hallucinations at scale.

[Building LLM Applications for Production](https://huyenchip.com/2023/04/11/llm-engineering.html) (Chip Huyen) -- one of the most referenced practical guides on the gap between demos and production.

---

*This document is part of the extras folder. Nothing here is required. All of it is real.*